# DEH 30-Day PySpark Challenge
## Day 30 — Full Mock Interview

**Rules:**
1. Set a 45-minute timer for all 3 problems
2. SQL first, then PySpark, for each problem
3. No other notebooks open, no searching
4. Talk through your approach in comments before coding
5. Reference solutions are at the bottom — only check after your timer ends

---

**Congratulations on reaching Day 30.** Good luck.

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder.appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id", StringType(), True), StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True), StructField("order_date", DateType(), True),
    StructField("quantity", IntegerType(), True), StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", IntegerType(), True), StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True), StructField("region", StringType(), True)
])
customers_schema = StructType([
    StructField("customer_id", StringType(), True), StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True), StructField("email", StringType(), True),
    StructField("city", StringType(), True), StructField("state", StringType(), True),
    StructField("country", StringType(), True), StructField("signup_date", DateType(), True),
    StructField("segment", StringType(), True)
])
products_schema = StructType([
    StructField("product_id", StringType(), True), StructField("product_name", StringType(), True),
    StructField("category", StringType(), True), StructField("sub_category", StringType(), True),
    StructField("unit_price", DoubleType(), True), StructField("cost_price", DoubleType(), True),
    StructField("supplier", StringType(), True), StructField("stock_quantity", IntegerType(), True)
])

orders_df    = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
customers_df = spark.read.option("header","true").schema(customers_schema).csv(f"s3a://{BUCKET}/data/customers.csv")
products_df  = spark.read.option("header","true").schema(products_schema).csv(f"s3a://{BUCKET}/data/products.csv")

orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")
products_df.createOrReplaceTempView("products")

print(f"Orders: {orders_df.count()} | Customers: {customers_df.count()} | Products: {products_df.count()}")
print("\nSet your 45-minute timer now. Good luck.")

---
# Problem 1 (~12 min) — Customer lifetime value tiering

Total spend per customer. Tier: Platinum >$3000, Gold >$1500, Silver >$500, else Bronze.  
Return customer_id, first_name, last_name, total_spend, tier. Sort by spend descending.

In [ ]:
# Your SQL solution


In [ ]:
# Your PySpark solution


---
# Problem 2 (~15 min) — Best-selling product per region per quarter

For each region + quarter combo, find the top product_name by total revenue.  
Return region, quarter, product_name, revenue.

In [ ]:
# Your SQL solution


In [ ]:
# Your PySpark solution


---
# Problem 3 (~18 min) — Customer churn risk report

At risk: last order > 60 days before the dataset's latest order date AND total_orders >= 3.  
Return customer_id, full_name, last_order_date, days_since_last_order, total_orders, total_spend.  
Write result to S3 as Parquet partitioned by segment.

In [ ]:
# Your SQL solution


In [ ]:
# Your PySpark solution (including the S3 write)


---
# Reference Solutions

**Only look below after your 45 minutes are up.**

---

## Reference Solution — Problem 1

In [ ]:
spark.sql("""
    WITH spend AS (
        SELECT customer_id, SUM(unit_price * quantity) AS total_spend
        FROM orders
        GROUP BY customer_id
    )
    SELECT c.customer_id, c.first_name, c.last_name,
           ROUND(s.total_spend, 2) AS total_spend,
           CASE
               WHEN s.total_spend > 3000 THEN 'Platinum'
               WHEN s.total_spend > 1500 THEN 'Gold'
               WHEN s.total_spend > 500  THEN 'Silver'
               ELSE 'Bronze'
           END AS tier
    FROM spend s
    INNER JOIN customers c ON s.customer_id = c.customer_id
    ORDER BY total_spend DESC
""").show(10)

In [ ]:
spend = orders_df.groupBy("customer_id").agg(
    F.round(F.sum(F.col("unit_price") * F.col("quantity")), 2).alias("total_spend")
)

spend.join(customers_df, on="customer_id", how="inner") \
    .withColumn(
        "tier",
        F.when(F.col("total_spend") > 3000, "Platinum")
         .when(F.col("total_spend") > 1500, "Gold")
         .when(F.col("total_spend") > 500,  "Silver")
         .otherwise("Bronze")
    ).select("customer_id", "first_name", "last_name", "total_spend", "tier") \
    .orderBy(F.col("total_spend").desc()) \
    .show(10)

## Reference Solution — Problem 2

In [ ]:
spark.sql("""
    WITH product_revenue AS (
        SELECT o.region, QUARTER(o.order_date) AS quarter, p.product_name,
               SUM(o.unit_price * o.quantity) AS revenue
        FROM orders o
        INNER JOIN products p ON o.product_id = p.product_id
        GROUP BY o.region, QUARTER(o.order_date), p.product_name
    ),
    ranked AS (
        SELECT *, RANK() OVER (PARTITION BY region, quarter ORDER BY revenue DESC) AS rnk
        FROM product_revenue
    )
    SELECT region, quarter, product_name, ROUND(revenue, 2) AS revenue
    FROM ranked
    WHERE rnk = 1
    ORDER BY region, quarter
""").show(20)

In [ ]:
product_revenue = orders_df.join(products_df, on="product_id", how="inner") \
    .withColumn("quarter", F.quarter("order_date")) \
    .groupBy("region", "quarter", "product_name").agg(
        F.sum(orders_df["unit_price"] * orders_df["quantity"]).alias("revenue")
    )

w = Window.partitionBy("region", "quarter").orderBy(F.col("revenue").desc())

product_revenue.withColumn("rnk", F.rank().over(w)) \
    .filter(F.col("rnk") == 1) \
    .select("region", "quarter", "product_name", F.round("revenue", 2).alias("revenue")) \
    .orderBy("region", "quarter") \
    .show(20)

## Reference Solution — Problem 3

In [ ]:
spark.sql("""
    WITH customer_orders AS (
        SELECT customer_id,
               MAX(order_date) AS last_order_date,
               COUNT(order_id) AS total_orders,
               SUM(unit_price * quantity) AS total_spend
        FROM orders
        GROUP BY customer_id
    ),
    latest_overall AS (
        SELECT MAX(order_date) AS max_date FROM orders
    )
    SELECT
        c.customer_id,
        CONCAT(c.first_name, ' ', c.last_name) AS full_name,
        c.segment,
        co.last_order_date,
        DATEDIFF(l.max_date, co.last_order_date) AS days_since_last_order,
        co.total_orders,
        ROUND(co.total_spend, 2) AS total_spend
    FROM customer_orders co
    INNER JOIN customers c ON co.customer_id = c.customer_id
    CROSS JOIN latest_overall l
    WHERE DATEDIFF(l.max_date, co.last_order_date) > 60
      AND co.total_orders >= 3
    ORDER BY days_since_last_order DESC
""").show(10)

In [ ]:
customer_orders = orders_df.groupBy("customer_id").agg(
    F.max("order_date").alias("last_order_date"),
    F.count("order_id").alias("total_orders"),
    F.round(F.sum(F.col("unit_price") * F.col("quantity")), 2).alias("total_spend")
)

max_date = orders_df.agg(F.max("order_date")).collect()[0][0]

churn_risk = customer_orders.join(customers_df, on="customer_id", how="inner") \
    .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name"))) \
    .withColumn("days_since_last_order", F.datediff(F.lit(max_date), F.col("last_order_date"))) \
    .filter((F.col("days_since_last_order") > 60) & (F.col("total_orders") >= 3)) \
    .select("customer_id", "full_name", "segment", "last_order_date",
            "days_since_last_order", "total_orders", "total_spend") \
    .orderBy(F.col("days_since_last_order").desc())

churn_risk.show(10)

# Write to S3 partitioned by segment
churn_risk.write.mode("overwrite").partitionBy("segment") \
    .parquet(f"s3a://{BUCKET}/output/churn_risk_report/")

verify = spark.read.parquet(f"s3a://{BUCKET}/output/churn_risk_report/")
print(f"\nRows written: {churn_risk.count()} | Rows verified: {verify.count()}")

---
## You're Done

If you completed all three problems within 45 minutes — even with mistakes — you're ready for real PySpark interviews.  
If you needed more time or had to reference earlier days, that's exactly the signal for where to spend more practice before your next interview.

Review the problems you struggled with most. Rebuild them from scratch tomorrow without looking at the solution.

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*

**Congratulations on completing the 30-Day PySpark Challenge.**